# **Expected Returns Model Analysis**

## Expected Returns Notebook for all calculated stock features

### Covers all pymc models from :
- `probabilistic_ml_model.pymc_models.EarningsBeatModel.EarningsBeatBayesian` — hierarchical Beta-Binomial beat-probability model with DB-aligned `pm.Data` containers (`n_total`, `n_beats`, `sector_idx`, `earnings_features`).
- `probabilistic_ml_model.pymc_models.PriceTargetModel.PriceTargetAchievement` — hierarchical Beta-Binomial beat-probability model with DB-aligned `pm.Data` containers (`n_total`, `n_beats`, `sector_idx`, `pt_features`).
- `probabilistic_ml_model.pymc_models.KalmanFilterModel.KalmanFilterPriceTarget` — hierarchical Beta-Binomial beat-probability model with DB-aligned `pm.Data` containers (`n_total`, `n_beats`, `sector_idx`, `kalman_features_df`).
- `public.mv_all_stock_features` — main dataframe for model analyis.
- `public.calculated_features_registry` (category = `Earnings`) — drives the `earnings_feature` coord labels.


In [1]:

import pandas as pd

try:
    import arviz as az
except ImportError:
    import arviz_base as az

# Public PyMC model aliases — resolved lazily by `probabilistic_ml_model/__init__.py`.
from probabilistic_ml_model import (
    EarningsBeatBayesian,
    PriceTargetAchievement,
    KalmanFilterPriceTarget,
    DCFPriceTarget,
    DividendSafetyBayesian,
    CreditRiskBayesian,
    AccountingAnomalyBayesian,
    # Shared `_feature_alignment` helpers (re-exported as public package aliases).
    assert_disjoint_features,
    coerce_by_data_type,
    load_feature_metadata_from_db,
    stamp_feature_provenance,
    validate_oos_shape,
)

# Per-model category-key tuples and `_resolve_*_feature_aliases` helpers are
# module/class internals (single-leading-underscore by convention). They are
# intentionally accessed via `getattr` below to make the protected-member
# coupling explicit and silence lint warnings without renaming the symbols.
from probabilistic_ml_model.pymc_models import (
    AccountingAnomalyModel as _anomaly_mod,
    PriceTargetModel as _pt_mod,
    KalmanFilterModel as _kalman_mod,
    EarningsBeatModel as _earnings_mod,
    DCF_PriceTargetModel as _dcf_mod,
    DividendSafetyModel as _dividend_mod,
    CreditRiskModel as _credit_mod,
)

_ANOMALY_CATEGORY_KEYS = getattr(_anomaly_mod, '_ANOMALY_CATEGORY_KEYS')
_PT_CATEGORY_KEYS = getattr(_pt_mod, '_PT_CATEGORY_KEYS')
_KALMAN_CATEGORY_KEYS = getattr(_kalman_mod, '_KALMAN_CATEGORY_KEYS')
_EARNINGS_CATEGORY_KEYS = getattr(_earnings_mod, '_EARNINGS_CATEGORY_KEYS')
_DCF_CATEGORY_KEYS = getattr(_dcf_mod, '_DCF_CATEGORY_KEYS')
_DIVIDEND_CATEGORY_KEYS = getattr(_dividend_mod, '_DIVIDEND_CATEGORY_KEYS')
_CREDIT_CATEGORY_KEYS = getattr(_credit_mod, '_CREDIT_CATEGORY_KEYS')


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


In [2]:
%%sql
SELECT * FROM postgres.pml.pml_df pd

,ticker,isin,name,description,region,country,trading_country,exchange,unit,sector,...,volatility_3m,volatility_6m,volatility_1y,eps_gaap_est_avg_rev_pct_fy1e_1w,eps_gaap_est_avg_rev_pct_fy1e_mtd,eps_gaap_est_avg_rev_pct_fy1e_qtd,eps_gaap_est_avg_rev_pct_fy1e_ytd,price_target_stddev_mtd_ago,price_target_stddev_qtd_ago,price_target_stddev_ytd_ago
0,NVDA,US67066G1040,NVIDIA Corporation,NVIDIA Corporation operates as a data center s...,United States and Canada,US,US,NasdaqGS,USD,Information Technology,...,35.97,35.02,33.41,0.0016,0.0016,0.0034,0.1612,42.4898,42.6325,39.8856
1,AAPL,US0378331005,Apple Inc.,Apple Inc. designs manufactures and markets sm...,United States and Canada,US,US,NasdaqGS,USD,Information Technology,...,23.28,22.40,22.48,-0.0042,0.0255,0.0270,0.0553,30.7828,31.2942,34.8281
2,BRVO,CA10566M1068,Bravo Mining Corp.,Bravo Mining Corp. acquires explores operates ...,Latin America and Caribbean,BR,CA,TSXV,CAD,Materials,...,76.35,85.60,76.60,NaN,NaN,NaN,NaN,1.1360,1.2766,1.3411
3,MSFT,US5949181045,Microsoft Corporation,Microsoft Corporation develops and supports so...,United States and Canada,US,US,NasdaqGS,USD,Information Technology,...,27.05,29.29,23.55,0.0003,0.0105,0.0064,0.1049,72.7010,67.8552,50.0584
4,LNZA,US51655R2004,LanzaTech Global Inc.,LanzaTech Global Inc. operates as a nature-bas...,United States and Canada,US,US,NasdaqCM,USD,Industrials,...,215.02,165.19,162.46,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6575,LOTON0000,MU0431N00000,Lottotech Ltd,Lottotech Ltd operates the Mauritius National ...,Africa / Middle East,MU,MU,MUSE,MUR,Consumer Discretionary,...,29.78,29.29,34.88,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000
6576,ASSAD,TN0007140015,L'Accumulateur Tunisien Assad SA,L'Accumulateur Tunisien Assad SA designs manuf...,Africa / Middle East,TN,TN,BVMT,TND,Industrials,...,36.74,60.34,59.79,0.0055,0.0052,0.0192,0.0017,0.0000,0.0000,0.0000
6577,BERGER,NGBERGER0000,Berger Paints Nigeria Plc,Berger Paints Nigeria Plc manufactures distrib...,Africa / Middle East,NG,NG,NGSE,NGN,Materials,...,110.07,92.83,107.29,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000
6578,SIAME,TN0006590012,Société Industrielle d'Appareillage et de Maté...,Société Industrielle d'Appareillage et de Maté...,Africa / Middle East,TN,TN,BVMT,TND,Industrials,...,35.37,39.59,43.50,0.0055,0.0052,0.0192,0.0017,0.0000,0.0000,0.0000


In [3]:
pml_df = pd.DataFrame(df)

In [4]:
pml_df_columns = pml_df.columns.tolist()
pml_df_columns

['ticker',
 'isin',
 'name',
 'description',
 'region',
 'country',
 'trading_country',
 'exchange',
 'unit',
 'sector',
 'industry',
 'style_class',
 'size_class',
 'last_updated',
 'income_statement_report_date',
 'fy_end',
 'next_earnings',
 'next_earnings_when',
 'next_earnings_status',
 'fy_end_date',
 'next_income_statement_report_date',
 'next_fy_end_date',
 'expected_report_date',
 'dividend_record_currency',
 'dividend_record_amount',
 'dividend_record_frequency',
 'dividend_streak',
 'dividend_record_announce_date',
 'dividend_record_payable_date',
 'dividend_record_record_date',
 'dividend_record_ex_date',
 'market_cap',
 'enterprise_value',
 'last_price',
 'price_target_ytd_ago',
 'total_return_ytd',
 'price_target',
 'price_target_low',
 'price_target_median',
 'price_target_high',
 'price_target_num',
 'p_e_ntm',
 'p_e_ltm',
 'altman_z_score_fy',
 'altman_z_score_fq',
 'altman_z_score_ltm',
 'beta_1y',
 'beta_2y',
 'beta_5y',
 'analyst_rating',
 'num_strong_sell_ratings',

In [5]:
pml_df.describe()

,dividend_record_amount,dividend_streak,market_cap,enterprise_value,last_price,price_target_ytd_ago,total_return_ytd,price_target,price_target_low,price_target_median,...,volatility_3m,volatility_6m,volatility_1y,eps_gaap_est_avg_rev_pct_fy1e_1w,eps_gaap_est_avg_rev_pct_fy1e_mtd,eps_gaap_est_avg_rev_pct_fy1e_qtd,eps_gaap_est_avg_rev_pct_fy1e_ytd,price_target_stddev_mtd_ago,price_target_stddev_qtd_ago,price_target_stddev_ytd_ago
count,4246.000000,4800.000000,6.580000e+03,6.569000e+03,6.580000e+03,6.401000e+03,6530.000000,6.580000e+03,6.580000e+03,6.580000e+03,...,6576.000000,6577.000000,6578.000000,5045.000000,5037.000000,4987.000000,4904.000000,6551.000000,6492.000000,6401.000000
mean,41.495600,2.400208,1.461870e+04,1.670103e+04,4.602894e+03,3.979507e+03,0.152124,5.190226e+03,3.452966e+03,5.218620e+03,...,49.702267,48.401878,47.527037,0.035105,0.035312,0.040709,0.111303,877.528029,722.912350,585.984212
std,403.181458,4.906294,1.162946e+05,1.175588e+05,5.546715e+04,5.083326e+04,0.515209,6.576809e+04,4.468489e+04,6.590271e+04,...,25.430755,44.961476,47.779795,1.608885,1.605545,0.488131,2.705345,11818.024990,8834.005024,7409.494460
min,0.001000,0.000000,1.028000e+01,-1.685380e+03,3.100000e-03,1.190000e-02,-0.741900,9.200000e-03,7.000000e-03,8.600000e-03,...,2.050000,1.840000,0.000000,-0.924100,-1.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000
25%,0.150000,1.000000,5.377250e+02,6.694300e+02,8.627500e+00,1.090000e+01,-0.102175,1.146565e+01,9.000000e+00,1.124000e+01,...,32.907500,31.340000,29.832500,-0.001100,-0.005500,-0.029450,-0.115550,0.553050,0.599750,0.635000
50%,0.500000,1.000000,2.662835e+03,3.019450e+03,2.928500e+01,3.381430e+01,0.046400,3.587380e+01,2.875000e+01,3.550000e+01,...,43.490000,41.300000,39.670000,0.003200,0.001000,0.009800,-0.012150,3.294400,3.386250,3.299800
75%,2.134500,2.000000,7.187488e+03,8.936500e+03,1.128250e+02,1.213846e+02,0.264325,1.282626e+02,1.010000e+02,1.283430e+02,...,60.480000,57.760000,56.217500,0.005600,0.007000,0.038450,0.069825,13.444300,13.135275,12.613600
max,14000.000000,54.000000,5.331514e+06,5.280370e+06,1.835000e+06,2.200992e+06,12.649200,2.180556e+06,1.700000e+06,2.200000e+06,...,319.950000,3031.990000,2855.310000,112.787100,112.600500,14.525400,177.303600,396277.361800,397377.657500,312132.689000


In [6]:
import random
import numpy as np
import pymc as pm

try:
    from pymc.sampling.jax import sample_blackjax_nuts, sample_numpyro_nuts

    _HAS_JAX_SAMPLERS = True
except Exception:  # pragma: no cover - JAX backend is optional
    sample_blackjax_nuts = sample_numpyro_nuts = None
    _HAS_JAX_SAMPLERS = False

import warnings

try:
    from pymc.exceptions import ImputationWarning

    # Show the per-column auto-imputation notice only once instead of spamming.
    warnings.filterwarnings("once", category=ImputationWarning)
except Exception:  # pragma: no cover - very old PyMC
    ImputationWarning = Warning

random.seed(42)
np.random.seed(42)

# --- Target numerical columns with non-trivial missingness ---------------------
# Schema-aligned with `pml.pml_df` (cross-referenced against the CREATE TABLE
# DDL in `sql_scripts/pml/pml_df.sql`). All names below are verified columns
# of `pml_df` — typos / placeholder aliases from a sibling schema
# (`mv_all_stock_features`) have been removed.
IMPUTE_COLS = [
    # --- Valuation multiples (LTM/NTM forms in pml_df) ---
    "p_e_ltm",
    "p_e_ntm",
    "p_b_ltm",
    "ev_ebitda_ltm",
    "ev_ebitda_ntm",
    "ev_sales_ltm",
    "ev_sales_ntm",
    # --- Dividend yields ---
    "div_yield_ltm",
    "div_yield_ind",
    "div_yield_ntm",
    # --- EPS estimates (normalised + GAAP, NTM + FY1E) ---
    "eps_norm_est_avg_ntm",
    "eps_norm_est_avg_fy1e",
    "eps_gaap_est_avg_ntm",
    "eps_gaap_est_avg_fy1e",
    # --- Profitability ---
    "return_on_assets_roa_pct_ltm",
    "return_on_assets_roa_pct_fy",
    "gross_profit_margin_pct_ltm",
    "gross_profit_margin_pct_fy",
    # --- Risk / distress ---
    "altman_z_score_ltm",
    "altman_z_score_fy",
    "beta_1y",
    "beta_5y",
    # --- Analyst / target ---
    "price_target",
    "price_target_num",
    "target_pct_avg",
    "analyst_rating",
    # --- Cash-flow primitives used by DCF / DividendSafety ---
    "fcf_ltm",
    "fcf_fy",
    "cfo_ltm",
    "cfo_fy",
    "common_dividends_paid_ltm",
    # --- Volatility (used by PriceTarget risk_penalty) ---
    "volatility_1m",
    "volatility_3m",
    "volatility_1y",
]

# Keep only columns actually present in df (defensive — never trust the
# upstream view to expose every pml_df column), preserving order, de-duped.
_seen: set[str] = set()
IMPUTE_COLS = [
    c for c in IMPUTE_COLS
    if c in df.columns and not (c in _seen or _seen.add(c))
]

# Hierarchical grouping column (sector is the densest non-null categorical).
GROUP_COL = "sector" if "sector" in df.columns else None

# Sanity checks — IMPUTE_COLS must not contain the id/group columns or
# duplicates, otherwise the §15.2 column selection produces a DataFrame
# with duplicate labels and pandas raises
# `ValueError: cannot reindex on an axis with duplicate labels`.
assert "isin" not in IMPUTE_COLS, "IMPUTE_COLS leaked the 'isin' id column."
if GROUP_COL is not None:
    assert GROUP_COL not in IMPUTE_COLS, (
        f"IMPUTE_COLS leaked the group column '{GROUP_COL}'."
    )
assert len(IMPUTE_COLS) == len(set(IMPUTE_COLS)), "IMPUTE_COLS has duplicates."

print(f"Imputing {len(IMPUTE_COLS)} columns over {len(df)} rows.")
print(df[IMPUTE_COLS].isna().mean().mul(100).round(2).rename("missing_%").to_frame())


Imputing 34 columns over 6580 rows.
                              missing_%
p_e_ltm                           26.90
p_e_ntm                           17.29
p_b_ltm                            5.97
ev_ebitda_ltm                     15.05
ev_ebitda_ntm                     13.05
ev_sales_ltm                       5.41
ev_sales_ntm                       6.41
div_yield_ltm                     38.15
div_yield_ind                     30.24
div_yield_ntm                     19.42
eps_norm_est_avg_ntm              15.30
eps_norm_est_avg_fy1e             16.38
eps_gaap_est_avg_ntm               7.14
eps_gaap_est_avg_fy1e              7.66
return_on_assets_roa_pct_ltm       5.52
return_on_assets_roa_pct_fy        0.90
gross_profit_margin_pct_ltm        6.09
gross_profit_margin_pct_fy         5.23
altman_z_score_ltm                20.29
altman_z_score_fy                 12.13
beta_1y                           17.81
beta_5y                           19.73
price_target                       0.00
pric

In [7]:
# Work on a sector-aware copy keyed by isin (matches every downstream model).
if GROUP_COL is not None:
    _imp_df = df[["isin", GROUP_COL] + IMPUTE_COLS].copy()
    _imp_df = _imp_df[_imp_df[GROUP_COL].notna()].reset_index(drop=True)
    sector_labels, sector_idx = np.unique(
        _imp_df[GROUP_COL].astype(str).to_numpy(), return_inverse=True
    )
    n_sectors = len(sector_labels)
else:
    _imp_df = df[["isin"] + IMPUTE_COLS].copy()
    sector_labels = np.array(["__global__"])
    sector_idx = np.zeros(len(_imp_df), dtype=int)
    n_sectors = 1

# --- Per-feature robust standardisation (median / MAD) ---------------------
# Heavy-tailed accounting features (assets_*, inventory_*, cfo_fy) have
# sample_std >> sample_mean, so unit-scale priors against raw values cause
# the posterior to collapse toward 0. We z-score with median/MAD (robust to
# outliers) here, fit on the standardised scale, and back-transform in 15.4.
_scaler = {}
for col in IMPUTE_COLS:
    x = _imp_df[col].astype(float).to_numpy()
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    scale = 1.4826 * mad if mad > 0 else np.nanstd(x)
    if not np.isfinite(scale) or scale == 0:
        scale = 1.0
    _scaler[col] = (med, scale)
    _imp_df[col] = (x - med) / scale

# Defensive guard: drop sectors that are entirely missing across every feature.
_present = (
    _imp_df[IMPUTE_COLS].notna()
    .groupby(sector_idx).any().any(axis=1)
)
_valid_sectors = _present[_present].index.to_numpy()
_keep = np.isin(sector_idx, _valid_sectors)
_imp_df = _imp_df.loc[_keep].reset_index(drop=True)
sector_idx = sector_idx[_keep]

# Build plain ndarray observations with NaN (modern PyMC detects missingness
# from NaN and avoids the deprecated MaskedArray code-path).
obs_arrays = {col: _imp_df[col].to_numpy(dtype=float) for col in IMPUTE_COLS}
_missing_mask = {col: np.isnan(obs_arrays[col]) for col in IMPUTE_COLS}

print(f"Sectors: {n_sectors} | Rows: {len(_imp_df)} | Features: {len(IMPUTE_COLS)}")

Sectors: 9 | Rows: 6580 | Features: 34


In [8]:
coords = {
    "feature": IMPUTE_COLS,
    "sector": sector_labels,
    "isin": _imp_df["isin"].astype(str).to_numpy(),
}

with warnings.catch_warnings():
    # Cosmetic: one ImputationWarning per column would otherwise spam output.
    warnings.simplefilter("ignore", ImputationWarning)

    with pm.Model(coords=coords) as imputation_model:
        # --- Hyperpriors per feature (data is z-scored => unit-scale priors) --
        mu_global = pm.Normal("mu_global", mu=0.0, sigma=1.0, dims="feature")
        sigma_sec = pm.HalfNormal("sigma_sec", sigma=0.5, dims="feature")
        sigma_obs = pm.HalfNormal("sigma_obs", sigma=1.0, dims="feature")
        nu = pm.Gamma("nu", alpha=2.0, beta=0.1, dims="feature")  # df ~ 20

        # Non-centred sector offsets: (feature, sector).
        z_sec = pm.Normal("z_sec", mu=0.0, sigma=1.0, dims=("feature", "sector"))
        mu_sector = pm.Deterministic(
            "mu_sector",
            mu_global[:, None] + sigma_sec[:, None] * z_sec,
            dims=("feature", "sector"),
        )

        # --- Student-T likelihood per feature ---------------------------------
        # Student-T tolerates the heavy tails of the accounting features. Passing
        # plain ndarrays with NaN lets PyMC auto-create latent "<col>_unobserved"
        # RVs without the deprecated MaskedArray code-path.
        imputed_vars = {}
        for j, col in enumerate(IMPUTE_COLS):
            imputed_vars[col] = pm.StudentT(
                col,
                nu=nu[j],
                mu=mu_sector[j, sector_idx],
                sigma=sigma_obs[j],
                observed=obs_arrays[col],
                dims="isin",
            )

        # Sampling -- prefer JAX NUTS when available.
        if _HAS_JAX_SAMPLERS:
            try:
                imputation_idata = sample_numpyro_nuts(
                    draws=500, tune=1000, chains=4,
                    target_accept=0.95, random_seed=42,
                )
            except Exception:
                imputation_idata = pm.sample(
                    draws=500, tune=1000, chains=4,
                    target_accept=0.95, random_seed=42, progressbar=False,
                )
        else:
            imputation_idata = pm.sample(
                draws=500, tune=1000, chains=4,
                target_accept=0.95, random_seed=42, progressbar=False,
            )

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: e(Sub, ~z, e(Mul, ~alpha [<function <lambda> at 0x000001CF20E35430>], e(SparseDot, ~x, ~y))) -> e(Usmm{no_inplace}, e(Neg, ~alpha [<function <lambda> at 0x000001CF20E35430>]), ~x, ~y, ~z)
ERROR (pytensor.graph.rewriting.basic): node: Sub([-4.60517019], Mul.0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "C:\Users\markm\PML_Finance_Project\.venv\Lib\site-packages\pytensor\configparser.py", line 299, in fetch_val_for_key
    return self._pytensor_cfg.get(section, option)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "C:\Users\markm\AppData\Local\Programs\Python\Python314\Lib\configparser.py", line 830, in get
    d = self._unify_values(section, vars)
  File "C:\Users\markm\AppData\Local\Programs\Python\Python314\Lib\configparser.py", line 1203, in _unify_values
    raise NoSectionError(section) from None
configparser.NoSection

KeyboardInterrupt: 